# Weekly Project 02 - Image Features

## Robot Tracking

It is recommended that you finish the exercises from Monday, before starting the project.

For this project you are given a video of some mobile robots (Robots.mp4). The task is now to track only the robots that are moving. Try to use both sparse and dense optical flow and compare the results.

For sparse optical flow, draw the tracked keypoints onto each frame and try to show the frames fast enough, such that it looks like a video.

For dense optical flow, represent the movement in any way you see fitting. For example by making a new image with the colors of each pixel representing the movement.



## Weekly Project 02 - Start Here

In [2]:
import cv2
import numpy as np
from matplotlib import pyplot as plt

First, we read and resize the video frames to improve processing speed. We then convert each frame to grayscale and store the resulting images in a list.

In [3]:
cap = cv2.VideoCapture("Robots.mp4")

scale = 0.5
frames = []

while True:
    ret, frame = cap.read()

    if not ret:
        break

    frame_gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    frame_gray = cv2.resize(
        frame_gray,
        None,
        fx=scale,
        fy=scale,
        interpolation=cv2.INTER_AREA
    )

    frames.append(frame_gray)

cap.release()

print(f"Stored {len(frames)} scaled frames")

Stored 482 scaled frames


This section simply plays the video in grayscale because all frames were stored as grayscale images. It is included for visual inspection only.

In [ ]:
for _ in range(len(frames)):
    cv2.imshow('image', frames[_])
    cv2.waitKey(20)
cv2.destroyAllWindows()

## Sparse optical flow

For sparse optical flow, we begin by using the first frame as the reference frame and detecting the features that will be tracked throughout the video. We select up to 100 of the strongest features. The qualityLevel is set to 0.3, meaning that a feature is accepted only if its quality is at least 30% of that of the strongest detected feature. We also require a minimum distance of seven pixels between features so that they are spread across the image.

We then create a blank image called trail, with the same dimensions as the first frame. This image stores the lines representing the paths followed by the tracked features.

The features are tracked by applying calcOpticalFlowPyrLK() to each pair of consecutive frames. The function returns a status value for every feature: 1 indicates successful tracking, while 0 indicates that tracking failed. 
Using these values, good_old stores the valid feature positions in the previous frame, and good_new stores their corresponding positions in the current frame.

Finally, each successfully tracked position is paired with its new position. A green line is drawn between them on the persistent trail image, allowing the movement of each feature to be visualized over time.

In [12]:
old_frame = frames[0]

feat1 = cv2.goodFeaturesToTrack(old_frame, maxCorners=100, qualityLevel=0.3, minDistance=7)

trail = np.zeros((*old_frame.shape, 3), dtype=np.uint8)

for i in range(1, len(frames)):
    new_frame = frames[i]

    feat2, status, error = cv2.calcOpticalFlowPyrLK(old_frame, new_frame, feat1, None)

    if feat2 is None:
        break

    good_old = feat1[status.ravel() == 1]
    good_new = feat2[status.ravel() == 1]

    image = cv2.cvtColor(new_frame, cv2.COLOR_GRAY2BGR)

    for old, new in zip(good_old, good_new):
        x1, y1 = old.ravel().astype(int)
        x2, y2 = new.ravel().astype(int)

        cv2.line(trail, (x1, y1), (x2, y2), (0, 255, 0), 2)
        cv2.circle(image, (x2, y2), 5, (0, 255, 0), -1)

    cv2.imshow("Sparse optical flow", cv2.add(image, trail))

    if cv2.waitKey(20) & 0xFF == ord("q"):
        break

    # Required for tracking into the next frame
    old_frame = new_frame
    feat1 = good_new.reshape(-1, 1, 2)

cv2.destroyAllWindows()

## Dense optical flow

In [11]:
old_frame = frames[0]

step = 30          # Larger = faster and fewer arrows
arrow_scale = 10   # Makes small movements visible
min_motion = 0.05

for i in range(1, len(frames)):
    new_frame = frames[i]

    flow = cv2.calcOpticalFlowFarneback(old_frame, new_frame, None, 0.5, 2, 11, 2, 5, 1.1, 0)

    image = cv2.cvtColor(new_frame, cv2.COLOR_GRAY2BGR)

    # Create sampled pixel coordinates without nested loops
    y, x = np.mgrid[
        step // 2:new_frame.shape[0]:step,
        step // 2:new_frame.shape[1]:step
    ].reshape(2, -1)

    dx, dy = flow[y, x].T
    moving = np.hypot(dx, dy) > min_motion

    for x1, y1, move_x, move_y in zip(
        x[moving], y[moving], dx[moving], dy[moving]
    ):
        x2 = int(x1 + move_x * arrow_scale)
        y2 = int(y1 + move_y * arrow_scale)

        cv2.arrowedLine(
            image,
            (int(x1), int(y1)),
            (x2, y2),
            (0, 255, 0),
            1,
            tipLength=0.3
        )

    cv2.imshow("Dense optical flow", image)

    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

    old_frame = new_frame

cv2.destroyAllWindows()

## Challenge (optional)

If you have finished the other tasks and still have some time left, try the following:

 - Pick one of the moving robots and track only that one.

 - Try the same for the other video (Challenge.mp4).

You can for example try to detect which robot is which by using a feature descriptor and matching method (SIFT, ORB, etc.).

In [10]:
old_frame = frames[0]

# Select one robot, then press Enter
x, y, w, h = map(
    int,
    cv2.selectROI(
        "Select one robot",
        cv2.cvtColor(old_frame, cv2.COLOR_GRAY2BGR),
        showCrosshair=True
    )
)

cv2.destroyWindow("Select one robot")

# Detect features only inside the selected robot
feature_mask = np.zeros_like(old_frame)
feature_mask[y:y+h, x:x+w] = 255

points = cv2.goodFeaturesToTrack(
    old_frame,
    maxCorners=30,
    qualityLevel=0.01,
    minDistance=7,
    mask=feature_mask
)

trail = np.zeros((*old_frame.shape, 3), dtype=np.uint8)
box = np.array([x, y, w, h], dtype=float)

for frame_index in range(1, len(frames)):
    new_frame = frames[frame_index]

    new_points, status, error = cv2.calcOpticalFlowPyrLK(
        old_frame, new_frame, points, None
    )

    if new_points is None:
        break

    good_old = points[status.ravel() == 1].reshape(-1, 2)
    good_new = new_points[status.ravel() == 1].reshape(-1, 2)

    if len(good_new) < 3:
        break

    # Move the bounding box using the median feature movement
    movement = np.median(good_new - good_old, axis=0)
    box[0] += movement[0]
    box[1] += movement[1]

    image = cv2.cvtColor(new_frame, cv2.COLOR_GRAY2BGR)

    for old, new in zip(good_old, good_new):
        x1, y1 = old.astype(int)
        x2, y2 = new.astype(int)

        cv2.line(trail, (x1, y1), (x2, y2), (0, 255, 0), 2)
        cv2.circle(image, (x2, y2), 4, (0, 255, 0), -1)

    bx, by, bw, bh = box.astype(int)
    cv2.rectangle(
        image,
        (bx, by),
        (bx + bw, by + bh),
        (0, 0, 255),
        2
    )

    cv2.imshow("Selected robot", cv2.add(image, trail))

    if cv2.waitKey(20) & 0xFF == ord("q"):
        break

    old_frame = new_frame
    points = good_new.reshape(-1, 1, 2)

cv2.destroyAllWindows()